In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Kidney Cancer"   
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
#   - keep architecture flexible, but preprocessing matches second snippet
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")


# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "cnn"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

2025-09-26 21:18:58.964313: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-26 21:18:59.010257: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-26 21:18:59.843302: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Found 10000 files belonging to 2 classes.
Using 8500 files for training.


2025-09-26 21:19:03.198720: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9559 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:3d:00.0, compute capability: 7.5
2025-09-26 21:19:03.200269: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 9559 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:b2:00.0, compute capability: 7.5
2025-09-26 21:19:03.201666: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 9559 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:da:00.0, compute capability: 7.5


Found 10000 files belonging to 2 classes.
Using 1500 files for validation.
Classes: ['kidney_normal', 'kidney_tumor']
Epoch 1/20


2025-09-26 21:19:05.342800: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inbaseline_cnn/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2025-09-26 21:19:15.528272: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 456 of 1000
2025-09-26 21:19:25.511350: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 968 of 1000
2025-09-26 21:19:26.580860: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] Shuffle buffer filled.
2025-09-26 21:19:27.029143: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8600
2025-09-26 21:19:27.540515: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7fad9b9507c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-26 21:

1063/1063 [==============================] - 59s 31ms/step - loss: 0.6961 - accuracy: 0.4955 - val_loss: 0.6960 - val_accuracy: 0.0000e+00
Epoch 2/20
1063/1063 [==============================] - 34s 30ms/step - loss: 0.6933 - accuracy: 0.4996 - val_loss: 0.7064 - val_accuracy: 0.0000e+00
Epoch 3/20
1063/1063 [==============================] - 34s 31ms/step - loss: 0.6933 - accuracy: 0.5025 - val_loss: 0.6867 - val_accuracy: 1.0000
Epoch 4/20
1063/1063 [==============================] - 33s 30ms/step - loss: 0.6933 - accuracy: 0.4989 - val_loss: 0.6884 - val_accuracy: 1.0000
Epoch 5/20
1063/1063 [==============================] - 33s 30ms/step - loss: 0.6932 - accuracy: 0.4995 - val_loss: 0.6905 - val_accuracy: 1.0000
Epoch 6/20
1063/1063 [==============================] - 34s 31ms/step - loss: 0.6934 - accuracy: 0.4867 - val_loss: 0.7078 - val_accuracy: 0.0000e+00
Epoch 7/20
1063/1063 [==============================] - 34s 31ms/step - loss: 0.6933 - accuracy: 0.4951 - val_loss: 0.6987 

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Kidney Cancer"   
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")



# RESNET
MODEL_NAME = "resnet"   
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Found 10000 files belonging to 2 classes.
Using 8500 files for training.
Found 10000 files belonging to 2 classes.
Using 1500 files for validation.
Classes: ['kidney_normal', 'kidney_tumor']
Epoch 1/20
1063/1063 [==============================] - 28s 22ms/step - loss: 0.6326 - accuracy: 0.6398 - val_loss: 0.3874 - val_accuracy: 0.8670
Epoch 2/20
1063/1063 [==============================] - 23s 21ms/step - loss: 0.5427 - accuracy: 0.7228 - val_loss: 0.2305 - val_accuracy: 0.9641
Epoch 3/20
1063/1063 [==============================] - 23s 21ms/step - loss: 0.5116 - accuracy: 0.7467 - val_loss: 0.4915 - val_accuracy: 0.7061
Epoch 4/20
1063/1063 [==============================] - 23s 21ms/step - loss: 0.4956 - accuracy: 0.7552 - val_loss: 0.3254 - val_accuracy: 0.8657
Epoch 5/20
1063/1063 [==============================] - 23s 21ms/step - loss: 0.4737 - accuracy: 0.7693 - val_loss: 0.2927 - val_accuracy: 0.8856
Epoch 6/20
1063/1063 [==============================] - 24s 21ms/step - loss: 0

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Kidney Cancer"   
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)


from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")



# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "vgg"  
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)


if MODEL_NAME == "vgg":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")


model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Found 10000 files belonging to 2 classes.
Using 8500 files for training.
Found 10000 files belonging to 2 classes.
Using 1500 files for validation.
Classes: ['kidney_normal', 'kidney_tumor']
Epoch 1/20
1063/1063 [==============================] - 27s 21ms/step - loss: 0.6436 - accuracy: 0.6300 - val_loss: 0.2099 - val_accuracy: 0.9934
Epoch 2/20
1063/1063 [==============================] - 23s 20ms/step - loss: 0.5489 - accuracy: 0.7175 - val_loss: 0.6388 - val_accuracy: 0.5971
Epoch 3/20
1063/1063 [==============================] - 23s 21ms/step - loss: 0.5061 - accuracy: 0.7532 - val_loss: 0.2955 - val_accuracy: 0.8936
Epoch 4/20
1063/1063 [==============================] - 23s 21ms/step - loss: 0.4947 - accuracy: 0.7553 - val_loss: 0.4467 - val_accuracy: 0.7500
Epoch 5/20
1063/1063 [==============================] - 23s 21ms/step - loss: 0.4775 - accuracy: 0.7711 - val_loss: 0.4164 - val_accuracy: 0.7832
resnet50  |  Test acc: 1.0000  |  Test loss: 0.1966


In [4]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Kidney Cancer"   
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)


from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")


MODEL_NAME = "inception"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)


if MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Found 10000 files belonging to 2 classes.
Using 8500 files for training.
Found 10000 files belonging to 2 classes.
Using 1500 files for validation.
Classes: ['kidney_normal', 'kidney_tumor']
Epoch 1/20
1063/1063 [==============================] - 27s 20ms/step - loss: 0.3392 - accuracy: 0.8480 - val_loss: 0.4485 - val_accuracy: 0.7912
Epoch 2/20
1063/1063 [==============================] - 20s 18ms/step - loss: 0.2166 - accuracy: 0.9113 - val_loss: 0.6064 - val_accuracy: 0.6941
Epoch 3/20
1063/1063 [==============================] - 20s 18ms/step - loss: 0.2079 - accuracy: 0.9180 - val_loss: 0.1492 - val_accuracy: 0.9455
Epoch 4/20
1063/1063 [==============================] - 20s 18ms/step - loss: 0.1987 - accuracy: 0.9226 - val_loss: 0.4235 - val_accuracy: 0.8218
Epoch 5/20
1063/1063 [==============================] - 20s 18ms/step - loss: 0.1912 - accuracy: 0.9252 - val_loss: 0.4336 - val_accuracy: 0.8112
Epoch 6/20
1063/1063 [==============================] - 20s 18ms/step - loss: 0

In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Kidney Cancer"   
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)


from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")


MODEL_NAME = "efficientnet"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)


if MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Found 10000 files belonging to 2 classes.
Using 8500 files for training.
Found 10000 files belonging to 2 classes.
Using 1500 files for validation.
Classes: ['kidney_normal', 'kidney_tumor']
Epoch 1/20


2025-09-26 21:36:08.892927: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inefficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


1063/1063 [==============================] - 24s 17ms/step - loss: 0.7164 - accuracy: 0.5126 - val_loss: 0.5606 - val_accuracy: 1.0000
Epoch 2/20
1063/1063 [==============================] - 17s 15ms/step - loss: 0.7144 - accuracy: 0.5000 - val_loss: 0.5114 - val_accuracy: 1.0000
Epoch 3/20
1063/1063 [==============================] - 17s 15ms/step - loss: 0.7108 - accuracy: 0.5065 - val_loss: 0.5675 - val_accuracy: 1.0000
Epoch 4/20
1063/1063 [==============================] - 17s 15ms/step - loss: 0.7161 - accuracy: 0.4942 - val_loss: 0.5982 - val_accuracy: 1.0000
Epoch 5/20
1063/1063 [==============================] - 17s 15ms/step - loss: 0.7104 - accuracy: 0.5113 - val_loss: 0.6893 - val_accuracy: 0.6609
efficientnetb0  |  Test acc: 1.0000  |  Test loss: 0.5608
